In [1]:
import sys
print("Kernel:", sys.version)
print("Executable:", sys.executable)

!{sys.executable} -m pip -q uninstall -y ultralytics
!rm -rf ~/.config/Ultralytics ~/.cache/ultralytics

!{sys.executable} -m pip -q install -U --no-cache-dir ultralytics opencv-python pyyaml lapx


Kernel: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Executable: /usr/bin/python3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.5 MB/s eta 0:00:00a 0:00:01


In [2]:
import numpy as np, cv2, ultralytics
print("numpy:", np.__version__)
print("cv2:", cv2.__version__)
print("ultralytics:", ultralytics.__version__)
print("ultralytics file:", ultralytics.__file__)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
numpy: 2.0.2
cv2: 4.12.0
ultralytics: 8.3.241
ultralytics file: /usr/local/lib/python3.12/dist-packages/ultralytics/__init__.py


In [3]:
MODEL_PATH = "/kaggle/input/format-2/best_yolo11.pt"
VIDEO_PATH = "/kaggle/input/video-test-1/IMG_0005.MP4"


In [4]:
import os
print("MODEL exists:", os.path.exists(MODEL_PATH), MODEL_PATH)
print("VIDEO exists:", os.path.exists(VIDEO_PATH), VIDEO_PATH)


MODEL exists: True /kaggle/input/format-2/best_yolo11.pt
VIDEO exists: True /kaggle/input/video-test-1/IMG_0005.MP4


In [5]:
import yaml

tracker_cfg = {
    "tracker_type": "bytetrack",
    "track_high_thresh": 0.25,
    "track_low_thresh": 0.05,
    "new_track_thresh": 0.35,
    "track_buffer": 200,
    "match_thresh": 0.8,
    "fuse_score": True
}

TRACKER_PATH = "/kaggle/working/custom_bytetrack.yaml"
with open(TRACKER_PATH, "w") as f:
    yaml.safe_dump(tracker_cfg, f, sort_keys=False)

print("✅ Saved tracker:", TRACKER_PATH)
print(open(TRACKER_PATH, "r").read())


✅ Saved tracker: /kaggle/working/custom_bytetrack.yaml
tracker_type: bytetrack
track_high_thresh: 0.25
track_low_thresh: 0.05
new_track_thresh: 0.35
track_buffer: 200
match_thresh: 0.8
fuse_score: true



In [7]:
import os, shutil, glob

OUTDIR = "/kaggle/working/export_yolo_bytetrack"
os.makedirs(OUTDIR, exist_ok=True)

MODEL_PATH = "/kaggle/input/format-2/best_yolo11.pt"      # your model
TRACKER_PATH = "/kaggle/working/custom_bytetrack.yaml"    # your bytetrack config

shutil.copy(MODEL_PATH, f"{OUTDIR}/best_yolo11.pt")
shutil.copy(TRACKER_PATH, f"{OUTDIR}/custom_bytetrack.yaml")

# Optional: copy latest tracked video
mp4s = sorted(glob.glob("runs/track/**/**/*.mp4", recursive=True))
if mp4s:
    shutil.copy(mp4s[-1], f"{OUTDIR}/tracked_output.mp4")

print("Export folder:", OUTDIR)
print(os.listdir(OUTDIR))


Export folder: /kaggle/working/export_yolo_bytetrack
['best_yolo11.pt', 'custom_bytetrack.yaml']


In [8]:
script = r'''
from ultralytics import YOLO

MODEL_PATH = "best_yolo11.pt"
VIDEO_PATH = "your_video.mp4"   # change this
TRACKER_PATH = "custom_bytetrack.yaml"

model = YOLO(MODEL_PATH)
for _ in model.track(
    source=VIDEO_PATH,
    tracker=TRACKER_PATH,
    imgsz=1280,
    vid_stride=1,
    conf=0.05,
    iou=0.5,
    max_det=1,
    persist=True,
    save=True,
    device=0,
    stream=True
):
    pass
print("Done. Check runs/track/")
'''
open(f"{OUTDIR}/run_track.py","w").write(script)
print("Added run_track.py")

# re-zip after adding
import shutil
zip_path = shutil.make_archive("/kaggle/working/yolo_bytetrack_package", "zip", OUTDIR)
print("ZIP updated:", zip_path)


Added run_track.py
ZIP updated: /kaggle/working/yolo_bytetrack_package.zip


In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_PATH)

for _ in model.track(
    source=VIDEO_PATH,
    tracker=TRACKER_PATH,
    imgsz=1280,
    vid_stride=1,
    conf=0.05,
    iou=0.5,
    max_det=1,
    persist=True,
    save=True,
    device=0,
    stream=True
):
    pass


In [ ]:
import os, shutil, glob

OUTDIR = "/kaggle/working/export_yolo_bytetrack"
os.makedirs(OUTDIR, exist_ok=True)

MODEL_PATH = "/kaggle/input/format-2/best_yolo11.pt"      # your model
TRACKER_PATH = "/kaggle/working/custom_bytetrack.yaml"    # your bytetrack config

shutil.copy(MODEL_PATH, f"{OUTDIR}/best_yolo11.pt")
shutil.copy(TRACKER_PATH, f"{OUTDIR}/custom_bytetrack.yaml")

# Optional: copy latest tracked video
mp4s = sorted(glob.glob("runs/track/**/**/*.mp4", recursive=True))
if mp4s:
    shutil.copy(mp4s[-1], f"{OUTDIR}/tracked_output.mp4")

print("Export folder:", OUTDIR)
print(os.listdir(OUTDIR))


In [ ]:
import ultralytics
from pathlib import Path

cfg_path = Path(ultralytics.__file__).parent / "cfg" / "default.yaml"
txt = cfg_path.read_text()

if "fuse_score:" not in txt:
    txt = txt.rstrip() + "\n\n# Patched to avoid missing attribute in some builds\nfuse_score: False\n"
    cfg_path.write_text(txt)
    print("✅ Patched:", cfg_path)
else:
    print("✅ fuse_score already exists")

print("contains fuse_score?", "fuse_score:" in cfg_path.read_text())


In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_PATH)

model.track(
    source=VIDEO_PATH,          # ✅ مرة واحدة فقط
    tracker=TRACKER_PATH,       # ✅ مسار كامل
    imgsz=1280,
    vid_stride=1,
    conf=0.05,
    iou=0.5,
    max_det=1,
    persist=True,
    save=True,
    device=0
)
